# Test code

In [ ]:
# ============================================================
# STANDALONE TEST: load the saved adapter in a fresh Colab session
# ============================================================
%pip install -q -U "bitsandbytes>=0.46.1" transformers peft accelerate

import os, re, json, html
import torch
from google.colab import drive
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

drive.mount("/content/drive")

WORK_DIR = "/content/drive/MyDrive/Customer-Support-AI"
ADAPTER_DIR = os.path.join(WORK_DIR, "qwen25_support_adapter")
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# 1. Use the EXACT system prompt the adapter was trained with (saved by the training notebook)
FALLBACK_PROMPT = (
    "You are a helpful, empathetic customer support agent. Reply briefly, politely and "
    "professionally to the customer's message. Do not invent order details, links or phone numbers."
)
meta_path = os.path.join(ADAPTER_DIR, "training_meta.json")
if os.path.isfile(meta_path):
    with open(meta_path) as f:
        meta = json.load(f)
    SYSTEM_PROMPT = meta["system_prompt"]
    MODEL_ID = meta.get("base_model", MODEL_ID)
    print("✓ System prompt loaded from training_meta.json")
else:
    SYSTEM_PROMPT = FALLBACK_PROMPT
    print("! training_meta.json not found - using the default training prompt")

# 2. Checks
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. In Colab: Runtime > Change runtime type > GPU.")
if not os.path.isfile(os.path.join(ADAPTER_DIR, "adapter_config.json")):
    raise FileNotFoundError(f"Adapter not found in {ADAPTER_DIR}. Check the path.")
print("GPU:", torch.cuda.get_device_name(0))

# 3. Load tokenizer + 4-bit base model + adapter
USE_BF16 = torch.cuda.get_device_capability(0)[0] >= 8
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto")
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()
print("✓ Model and adapter loaded")

# 4. Same input cleaning as training
URL_RE = re.compile(r"https?://\S+|www\.\S+", re.I)
MENTION_RE = re.compile(r"@\w+")
SIGN_RES = [
    re.compile(r"\s*[\^~]\s?[A-Za-z]{1,12}\s*$"),
    re.compile(r"\s+[-–—]\s?[A-Z][A-Za-z]{1,11}\s*$"),
    re.compile(r"\s+/\s?[A-Za-z]{1,5}\s*$"),
]

def clean_input(t):
    t = html.unescape(str(t))
    t = URL_RE.sub(" ", t)
    t = MENTION_RE.sub(" ", t)
    t = t.replace("#", " ")
    t = re.sub(r"\s+", " ", t).strip()
    for _ in range(2):
        for rx in SIGN_RES:
            t = rx.sub("", t)
    return re.sub(r"^[\s,:;.\-]+", "", t).strip()

def build_prompt(customer):
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": customer}],
        tokenize=False,
        add_generation_prompt=True,
    )

def ask_support(message, max_tokens=120, sample=False):
    message = clean_input(message)
    if not message:
        return "Please tell me what you need help with."
    inputs = tokenizer(build_prompt(message), return_tensors="pt", add_special_tokens=False).to(model.device)
    kwargs = dict(max_new_tokens=max_tokens, repetition_penalty=1.1,
                  pad_token_id=tokenizer.pad_token_id, use_cache=True)
    kwargs.update(dict(do_sample=True, temperature=0.7, top_p=0.9) if sample else dict(do_sample=False))
    with torch.no_grad():
        out = model.generate(**inputs, **kwargs)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# 5. Predefined tests
test_cases = [
    "My order hasn't arrived and it's been 10 days. This is frustrating.",
    "How do I reset my password? The link in the email isn't working.",
    "I was charged twice for the same purchase. I want a refund.",
    "Can you give me the phone number of your manager and my order tracking link?",
    "My product arrived damaged. What should I do?",
    "Thank you for helping me resolve my issue.",
]
print("\n" + "=" * 60 + "\nPREDEFINED TEST CASES\n" + "=" * 60)
for i, msg in enumerate(test_cases, 1):
    print(f"\nTest {i}\nCustomer: {msg}\nAI Agent: {ask_support(msg)}\n" + "-" * 60)

# 6. Interactive chat (sampling on, so replies are more varied)
print("\n" + "=" * 60 + "\nINTERACTIVE CHAT - type 'exit' or 'quit' to stop\n" + "=" * 60)
while True:
    try:
        user_input = input("\nYou: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\nChat ended.")
        break
    if user_input.lower() in ("exit", "quit"):
        print("Chat ended.")
        break
    if user_input:
        print("Agent:", ask_support(user_input, sample=True))

Mounted at /content/drive


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Customer support model is ready!

Example 1: My order has not arrived. What should I do?
AI: We're sorry to hear this! Please reach out to us here: and we'll be happy to help.

Example 2: How can I reset my password?
AI: Hi there! Please check your DMs for further instructions.

Example 3: My product arrived damaged. Can you help?
AI: Hi there! We're here to help. Please DM us your tracking number and we'll take a look at this for you.

Ask your own customer-support question.
Type 'quit' or 'exit' to stop.

You: quit
Chat ended. Goodbye!
